In [10]:
# !pip install yfinance crewai crewai-tools

In [11]:
import re
import json
import os
import yfinance as yf
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import CodeInterpreterTool, FileReadTool

In [12]:
from pydantic import BaseModel, Field

class QueryAnalysisOutput(BaseModel):
    """Structured output for the query analysis task."""
    symbol: str = Field(..., description="Stock ticker symbol (e.g., TSLA, AAPL).")
    timeframe: str = Field(..., description="Time period (e.g., '1d', '1mo', '1y').")
    action: str = Field(..., description="Action to be performed (e.g., 'fetch', 'plot').")

In [13]:
llm = LLM(
    model="ollama/deepseek-r1:latest",
    base_url="http://localhost:11434",
    # temperature=0.7
)

# 1) Query parser agent
query_parser_agent = Agent(
    role="Stock Data Analyst",
    goal="Extract stock details and fetch required data from this user query: {query}.",
    backstory="You are a financial analyst specializing in stock market data retrieval.",
    llm=llm,
    verbose=True,
    memory=True,
)

query_parsing_task = Task(
    description="Analyze the user query and extract stock details.",
    expected_output="A dictionary with keys: 'symbol', 'timeframe', 'action'.",
    output_pydantic=QueryAnalysisOutput,
    agent=query_parser_agent,
)


# 2) Code writer agent
code_writer_agent = Agent(
    role="Senior Python Developer",
    goal="Write Python code to visualize stock data.",
    backstory="""You are a Senior Python developer specializing in stock market data visualization. 
                 You are also a Pandas, Matplotlib and yfinance library expert.
                 You are skilled at writing production-ready Python code""",
    llm=llm,
    verbose=True,
)

code_writer_task = Task(
    description="""Write Python code to visualize stock data based on the inputs from the stock analyst
                   where you would find stock symbol, timeframe and action.""",
    expected_output="A clean and executable Python script file (.py) for stock visualization.",
    agent=code_writer_agent,
)


# 3) Code interpreter agent (uses code interpreter tool from crewai)
code_interpreter_tool = CodeInterpreterTool()

code_execution_agent = Agent(
    role="Senior Code Execution Expert",
    goal="Review and execute the generated Python code by code writer agent to visualize stock data.",
    backstory="You are a code execution expert. You are skilled at executing Python code.",
    # tools=[code_interpreter_tool],
    allow_code_execution=True,   # This automatically adds the CodeInterpreterTool
    llm=llm,
    verbose=True,
)

code_execution_task = Task(
    description="""Review and execute the generated Python code by code writer agent to visualize stock data.""",
    expected_output="A clean and executable Python script file (.py) for stock visualization.",
    agent=code_execution_agent,
)


In [14]:
### --- CREW SETUP --- ###

crew = Crew(
    agents=[query_parser_agent, code_writer_agent, code_execution_agent],
    tasks=[query_parsing_task, code_writer_task, code_execution_task],
    process=Process.sequential
)

# Run the crew with an example query
result = crew.kickoff(inputs={"query": "Plot YTD stock gain of Tesla"})

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Data Analyst                                                                                      │
│                                                                                                                 │
│  Task: Analyze the user query and extract stock details.                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Stock Data Analyst                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  <think>                                                                                                        │
│  Okay, let me process this user query about Tesla's YTD stock gain. The symbol here is clearly "Tesla" from     │
│  finance contexts and common knowledge - I know TSLA is their ticker.                                           │
│                                                                                                                 │
│  For timeframe, "Year To Date" specifically refers to the current calendar year period, so that should be       │
│  straightforward mapping to a timeframe parameter.                                                              │
│                                                                                                                 │
│  The action requested is unambiguous too: plotting requires me to fetch historical price data covering this     │
│  entire period. Looking at my tools, there's definitely an option for getting time series stock prices.         │
│                                                                                                                 │
│  Hmm, I wonder if I'm missing anything here. The user seems to know exactly what they want - no additional      │
│  context or options needed. Just need to confirm TSLA as symbol, YTD as timeframe based on standard financial   │
│  abbreviations, and "plot" clearly indicating the data action required.                                         │
│  </think>                                                                                                       │
│  {                                                                                                              │
│    "symbol": "TSLA",                                                                                            │
│    "timeframe": "Year To Date",                                                                                 │
│    "action": "plot"                                                                                             │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Python Developer                                                                                 │
│                                                                                                                 │
│  Task: Write Python code to visualize stock data based on the inputs from the stock analyst                     │
│                     where you would find stock symbol, timeframe and action.                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Python Developer                                                                                 │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  <think>                                                                                                        │
│  Okay, let's process this stock visualization request. The user provided a JSON input with three keys: symbol,  │
│  timeframe, and action.                                                                                         │
│                                                                                                                 │
│  First, I need to validate these inputs because incorrect or missing data could lead to failed visualizations   │
│  or misleading results. For the symbol field, "TSLA" is clearly Tesla's ticker on Yahoo Finance - that seems    │
│  straightforward.                                                                                               │
│                                                                                                                 │
│  For timeframe validation, it should be one of the standard intervals for stock charts: 1d (daily), 5d          │
│  (weekly-ish), 1m (monthly), 6m (6 months), ytd (year-to-date), or max (all available data). The user           │
│  specified "Year To Date", which maps cleanly to 'ytd'.                                                         │
│                                                                                                                 │
│  The action field has two options here: 'plot' for a line chart showing price movements over time, and          │
│  'candlestick' for detailed OHLCV candle charts. Since the timeframe is set to year-to-date, I'll use 'plot'    │
│  as it's more appropriate for tracking cumulative performance.                                                  │
│                                                                                                                 │
│  I also need to consider potential errors - what if the user provides an invalid symbol or timeframe? The       │
│  response should be informative but not panic-inducing. Maybe suggest common symbols like AAPL, MSFT while      │
│  keeping a calm tone?                                                                                           │
│                                                                                                                 │
│  The data source is yfinance via its Python library. I'll use start_date and end_date parameters with the       │
│  current year's information to get exactly what we need for YTD visualization.                                  │
│                                                                                                                 │
│  For the final output format, JSON seems logical because it can structure both the validation feedback and the  │
│  actual chart data cleanly.                                                                                     │
│  </think>                                                                                                       │
│  ### Thought:                                                                                                   │
│                                                                                                                 │
│  Based on the user input, they want a stock chart for Tesla (TSLA) showing performance from January 1st of the  │
│  current year to today. This timeframe is appropriate f

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Code Execution Expert                                                                            │
│                                                                                                                 │
│  Task: Review and execute the generated Python code by code writer agent to visualize stock data.               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Senior Code Execution Expert                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  <｜begin▁of▁sentence｜>JSON: {                                                                                 │
│    "symbol": "TSLA",                                                                                            │
│    "timeframe": "Year To Date"                                                                                  │
│  }                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [15]:
print(result.raw)

<｜begin▁of▁sentence｜>JSON: {
  "symbol": "TSLA",
  "timeframe": "Year To Date"
}
